In [1]:
from __future__ import annotations

# =========================================================
# PS S6E4 - 018 family + synthetic-threshold-min
#
# Suggested experiment id:
#   exp_20260420_055_cat018_threshold_min_from_family
#
# Purpose:
# - Keep 018 family skeleton as faithfully as possible
# - Add only a minimal synthetic-threshold block derived from the
#   original-data discussion
# - Use 046b-style 3-stage bias search instead of 018 greedy bias tuning
#
# References used for reconstruction:
# - 018 notebook structure: CatBoost + pairwise TE + orig merge + bias from shared fileciteturn14file1
# - 046b bias search: 3-stage log-prob-space bias refine fileciteturn14file0
# =========================================================

In [2]:
import os
import gc
import json
import warnings
import itertools
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
import torch

warnings.filterwarnings("ignore")

In [3]:
# ---------------------------------------------------------
# CFG
# ---------------------------------------------------------
class CFG:
    COMP_NAME = "playground-series-s6e4"
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    EXP_ID = "exp_20260420_055_cat018_threshold_min_from_family"
    EXP_NAME = "cat_pairwise_te_threshold_min_from_family"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    SEED = 42
    N_SPLITS = 5
    TE_INNER_CV = 3

    ORIG_ROW_WEIGHT = 0.35

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {v: k for k, v in TARGET_MAP.items()}

    CATS = [
        "Soil_Type",
        "Crop_Type",
        "Crop_Growth_Stage",
        "Season",
        "Irrigation_Type",
        "Water_Source",
        "Mulching_Used",
        "Region",
    ]

    NUMS = [
        "Soil_pH",
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Temperature_C",
        "Humidity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    ALL_BASE = NUMS + CATS

    CATBOOST_PARAMS = {
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 8,
        "grow_policy": "SymmetricTree",
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "random_seed": SEED,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
        "devices": "0",
        "verbose": 500,
        "early_stopping_rounds": 150,
    }

    LOGIT_EPS = 1e-12

In [4]:
# ---------------------------------------------------------
# Utility
# ---------------------------------------------------------
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))


def apply_bias(proba, bias):
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)
    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)


def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)


def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0
    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias
    return best_bias, float(best_score)

In [5]:
# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
COMP_DIR = Path(f"/kaggle/input/competitions/{CFG.COMP_NAME}")
ORIG_PATH = Path("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv")

train_raw = pd.read_csv(COMP_DIR / "train.csv").dropna(subset=[CFG.TARGET_COL]).reset_index(drop=True)
test_raw = pd.read_csv(COMP_DIR / "test.csv").reset_index(drop=True)
orig_raw = pd.read_csv(ORIG_PATH).reset_index(drop=True)

print(f"train_raw: {train_raw.shape}")
print(f"test_raw : {test_raw.shape}")
print(f"orig_raw : {orig_raw.shape}")
print(train_raw.head())

train_raw: (630000, 21)
test_raw : (270000, 20)
orig_raw : (10000, 20)
   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1      

In [6]:
# ---------------------------------------------------------
# Feature engineering helpers (018 skeleton)
# ---------------------------------------------------------
def add_physical_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["ET_proxy"] = (d["Temperature_C"] * d["Wind_Speed_kmh"] * d["Sunlight_Hours"]) / (d["Humidity"] + 1.0)
    d["water_deficit"] = d["Soil_Moisture"] - (d["Rainfall_mm"] + d["Previous_Irrigation_mm"]) * 0.1
    d["soil_quality"] = d["Organic_Carbon"] / (d["Electrical_Conductivity"] + 0.1)
    return d


def add_threshold_min_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["soil_lt_25"] = (d["Soil_Moisture"] < 25).astype("int8")
    d["temp_gt_30"] = (d["Temperature_C"] > 30).astype("int8")
    d["rain_lt_300"] = (d["Rainfall_mm"] < 300).astype("int8")
    d["wind_gt_10"] = (d["Wind_Speed_kmh"] > 10).astype("int8")

    high_score = (
        2 * d["soil_lt_25"]
        + 2 * d["rain_lt_300"]
        + d["temp_gt_30"]
        + d["wind_gt_10"]
    )
    low_score = (
        2 * (d["Crop_Growth_Stage"].eq("Harvest").astype("int8"))
        + 2 * (d["Crop_Growth_Stage"].eq("Sowing").astype("int8"))
        + (d["Mulching_Used"].eq("Yes").astype("int8"))
    )

    d["deotte_high_score"] = high_score.astype(np.int32)
    d["deotte_low_score"] = low_score.astype(np.int32)
    d["deotte_score"] = (high_score - low_score).astype(np.int32)
    d["score_band"] = pd.cut(
        d["deotte_score"],
        bins=[-999, 0, 3, 999],
        labels=["low_band", "mid_band", "high_band"],
        include_lowest=True,
    ).astype(str)
    return d


def build_orig_target_prior_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    base_cols: list[str],
    target_col: str,
    target_map: dict,
):
    train_out = train_df.copy()
    test_out = test_df.copy()
    orig_out = orig_df.copy()

    orig_target_numeric = orig_out[target_col].map(target_map).astype(np.float32)

    for col in base_cols:
        feat_name = f"TE_ORIG_{col}"

        if orig_out[col].dtype == "object":
            mapping = orig_target_numeric.groupby(orig_out[col]).mean().to_dict()
            fallback = float(orig_target_numeric.mean())
            train_out[feat_name] = train_out[col].map(mapping).fillna(fallback).astype(np.float32)
            test_out[feat_name] = test_out[col].map(mapping).fillna(fallback).astype(np.float32)
            orig_out[feat_name] = orig_out[col].map(mapping).fillna(fallback).astype(np.float32)
        else:
            edges = np.histogram_bin_edges(orig_out[col].dropna(), bins=10)
            bucket_means = orig_target_numeric.groupby(pd.cut(orig_out[col], bins=edges, include_lowest=True)).mean().values
            fallback = float(orig_target_numeric.mean())

            def map_bins(v):
                if pd.isna(v):
                    return fallback
                idx = np.digitize(v, edges) - 1
                idx = min(max(idx, 0), len(bucket_means) - 1)
                val = bucket_means[idx]
                if pd.isna(val):
                    return fallback
                return float(val)

            train_out[feat_name] = train_out[col].apply(map_bins).astype(np.float32)
            test_out[feat_name] = test_out[col].apply(map_bins).astype(np.float32)
            orig_out[feat_name] = orig_out[col].apply(map_bins).astype(np.float32)

    return train_out, test_out, orig_out


def build_pairwise_combinations(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    source_cols: list[str],
):
    train_out = train_df.copy()
    test_out = test_df.copy()
    orig_out = orig_df.copy()

    pair_cols = []
    raw_pair_count = 0

    for left, right in combinations(source_cols, 2):
        raw_pair_count += 1
        feat_name = f"{left}___{right}"

        tr_token = train_out[left].astype(str) + "_" + train_out[right].astype(str)
        te_token = test_out[left].astype(str) + "_" + test_out[right].astype(str)
        og_token = orig_out[left].astype(str) + "_" + orig_out[right].astype(str)

        combined = pd.concat([tr_token, te_token, og_token], ignore_index=True)
        codes, _ = pd.factorize(combined)

        if pd.Series(codes).nunique() <= (len(combined) // 2):
            train_out[feat_name] = codes[:len(train_out)].astype(np.int32)
            test_out[feat_name] = codes[len(train_out):len(train_out) + len(test_out)].astype(np.int32)
            orig_out[feat_name] = codes[len(train_out) + len(test_out):].astype(np.int32)
            pair_cols.append(feat_name)

    return train_out, test_out, orig_out, pair_cols, raw_pair_count

In [7]:
# ---------------------------------------------------------
# Build shared-style features + threshold-min
# ---------------------------------------------------------
train = add_physical_features(train_raw)
test = add_physical_features(test_raw)
orig = add_physical_features(orig_raw)

train = add_threshold_min_features(train)
test = add_threshold_min_features(test)
orig = add_threshold_min_features(orig)

THRESHOLD_MIN_COLS = [
    "soil_lt_25",
    "temp_gt_30",
    "rain_lt_300",
    "wind_gt_10",
    "deotte_high_score",
    "deotte_low_score",
    "deotte_score",
    "score_band",
]

ALL_BASE_PLUS = CFG.ALL_BASE + THRESHOLD_MIN_COLS

train, test, orig = build_orig_target_prior_features(
    train_df=train,
    test_df=test,
    orig_df=orig,
    base_cols=ALL_BASE_PLUS,
    target_col=CFG.TARGET_COL,
    target_map=CFG.TARGET_MAP,
)

train, test, orig, pair_cols, raw_pair_count = build_pairwise_combinations(
    train_df=train,
    test_df=test,
    orig_df=orig,
    source_cols=ALL_BASE_PLUS,
)

for c in CFG.CATS:
    train[c] = train[c].astype(str)
    test[c] = test[c].astype(str)
    orig[c] = orig[c].astype(str)

# threshold categorical columns as strings for CatBoost handling consistency
for c in ["score_band"]:
    train[c] = train[c].astype(str)
    test[c] = test[c].astype(str)
    orig[c] = orig[c].astype(str)

y_train_full = train[CFG.TARGET_COL].map(CFG.TARGET_MAP).values.astype(int)
y_orig_full = orig[CFG.TARGET_COL].map(CFG.TARGET_MAP).values.astype(int)

features = [c for c in train.columns if c not in [CFG.TARGET_COL, CFG.ID_COL]]

print(f"raw_pair_count      = {raw_pair_count}")
print(f"accepted_pair_count = {len(pair_cols)}")
print(f"total_features      = {len(features)}")
print(pair_cols[:10])

raw_pair_count      = 351
accepted_pair_count = 315
total_features      = 372
['Soil_pH___Organic_Carbon', 'Soil_pH___Electrical_Conductivity', 'Soil_pH___Temperature_C', 'Soil_pH___Sunlight_Hours', 'Soil_pH___Wind_Speed_kmh', 'Soil_pH___Field_Area_hectare', 'Soil_pH___Soil_Type', 'Soil_pH___Crop_Type', 'Soil_pH___Crop_Growth_Stage', 'Soil_pH___Season']


In [8]:
# ---------------------------------------------------------
# Adversarial validation (018 diagnostic)
# ---------------------------------------------------------
X_av = pd.concat(
    [
        train_raw.drop([CFG.ID_COL, CFG.TARGET_COL], axis=1),
        orig_raw.drop([CFG.TARGET_COL], axis=1),
    ],
    axis=0,
).reset_index(drop=True)

y_av = np.array([0] * len(train_raw) + [1] * len(orig_raw))

for c in CFG.CATS:
    X_av[c] = X_av[c].astype(str)

av_model = CatBoostClassifier(
    iterations=150,
    learning_rate=0.1,
    random_seed=CFG.SEED,
    verbose=0,
    task_type="GPU" if torch.cuda.is_available() else "CPU",
)

av_model.fit(X_av, y_av, cat_features=CFG.CATS)
av_auc = roc_auc_score(y_av, av_model.predict_proba(X_av)[:, 1])
print(f"adversarial_auc = {av_auc:.6f}")

del X_av, y_av, av_model
gc.collect()

adversarial_auc = 0.694620


0

In [9]:
# ---------------------------------------------------------
# CV with in-fold Target Encoding on pairwise columns
# ---------------------------------------------------------
skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

oof_probs = np.zeros((len(train), 3), dtype=np.float32)
test_probs_sum = np.zeros((len(test), 3), dtype=np.float32)
fold_scores_raw = []
importances = []
final_feature_names = None

# additional cat columns for CatBoost final matrices
extra_cat_cols = [c for c in THRESHOLD_MIN_COLS if c in features]
cat_cols_for_pool = CFG.CATS + [c for c in extra_cat_cols if c not in CFG.CATS]

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y_train_full), 1):
    X_tr_fold = train.iloc[tr_idx][features].copy()
    y_tr_fold = y_train_full[tr_idx]

    X_va_fold = train.iloc[va_idx][features].copy()
    y_va_fold = y_train_full[va_idx]

    # shared-style: original merged only into outer train
    X_tr_merged = pd.concat([X_tr_fold, orig[features]], axis=0).reset_index(drop=True)
    y_tr_merged = np.concatenate([y_tr_fold, y_orig_full])

    encoder = TargetEncoder(
        target_type="multiclass",
        cv=CFG.TE_INNER_CV,
        random_state=CFG.SEED,
    )

    enc_tr = encoder.fit_transform(X_tr_merged[pair_cols], y_tr_merged).astype(np.float32)
    enc_va = encoder.transform(X_va_fold[pair_cols]).astype(np.float32)
    enc_te = encoder.transform(test[features][pair_cols]).astype(np.float32)

    X_tr_final = X_tr_merged.drop(columns=pair_cols).copy()
    X_va_final = X_va_fold.drop(columns=pair_cols).copy()
    X_te_final = test[features].drop(columns=pair_cols).copy()

    te_col_names = [f"TE_pair_{i}" for i in range(enc_tr.shape[1])]
    X_tr_final[te_col_names] = enc_tr
    X_va_final[te_col_names] = enc_va
    X_te_final[te_col_names] = enc_te

    if final_feature_names is None:
        final_feature_names = X_tr_final.columns.tolist()

    sw = compute_sample_weight("balanced", y_tr_merged).astype(np.float32)
    sw[len(tr_idx):] *= CFG.ORIG_ROW_WEIGHT

    model = CatBoostClassifier(**CFG.CATBOOST_PARAMS)
    model.fit(
        Pool(X_tr_final, y_tr_merged, cat_features=cat_cols_for_pool, weight=sw),
        eval_set=Pool(X_va_final, y_va_fold, cat_features=cat_cols_for_pool),
    )

    oof_probs[va_idx] = model.predict_proba(X_va_final)
    test_probs_sum += model.predict_proba(X_te_final)
    importances.append(model.get_feature_importance())

    fold_ba = balanced_accuracy_score(y_va_fold, oof_probs[va_idx].argmax(axis=1))
    fold_scores_raw.append(float(fold_ba))
    print(f"fold {fold}/{CFG.N_SPLITS} raw_ba = {fold_ba:.6f}")

    del X_tr_final, X_va_final, X_te_final, enc_tr, enc_va, enc_te, model
    gc.collect()

raw_cv_ba = float(balanced_accuracy_score(y_train_full, oof_probs.argmax(axis=1)))
print(f"raw_cv_ba = {raw_cv_ba:.6f}")

0:	learn: 0.9735972	test: 0.9523869	best: 0.9523869 (0)	total: 6.03s	remaining: 5h 1m 27s
bestTest = 0.9635207909
bestIteration = 113
Shrink model to first 114 iterations.
fold 1/5 raw_ba = 0.974783
0:	learn: 0.9735564	test: 0.9613617	best: 0.9613617 (0)	total: 64.6ms	remaining: 3m 13s
500:	learn: 0.9847006	test: 0.9677433	best: 0.9678325 (498)	total: 33.1s	remaining: 2m 44s
1000:	learn: 0.9894817	test: 0.9722166	best: 0.9722568 (999)	total: 1m 5s	remaining: 2m 10s
1500:	learn: 0.9908346	test: 0.9734918	best: 0.9735034 (1498)	total: 1m 37s	remaining: 1m 37s
2000:	learn: 0.9913346	test: 0.9743062	best: 0.9744382 (1993)	total: 2m 9s	remaining: 1m 4s
2500:	learn: 0.9917297	test: 0.9745744	best: 0.9747224 (2383)	total: 2m 40s	remaining: 32.1s
bestTest = 0.9747223986
bestIteration = 2383
Shrink model to first 2384 iterations.
fold 2/5 raw_ba = 0.976904
0:	learn: 0.9728095	test: 0.9422972	best: 0.9422972 (0)	total: 65.2ms	remaining: 3m 15s
500:	learn: 0.9846818	test: 0.9677229	best: 0.968005

In [10]:
# ---------------------------------------------------------
# 046b-style 3-stage bias search
# ---------------------------------------------------------
coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_probs, y_train_full,
    [coarse_vals, coarse_vals, coarse_vals]
)
print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_probs, y_train_full,
    [local_vals_0, local_vals_1, local_vals_2]
)
print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_probs, y_train_full,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2]
)
print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_probs_biased = apply_bias(oof_probs, final_bias)

stage1 best_bias: (np.float64(-1.0), np.float64(-0.9), np.float64(0.19999999999999973)) score: 0.9785003035052237
stage2 best_bias: (np.float64(-1.04), np.float64(-0.98), np.float64(0.09999999999999973)) score: 0.9785362393334003
stage3 best_bias: (np.float64(-1.06), np.float64(-0.99), np.float64(0.08999999999999973)) score: 0.9785391983258066


In [11]:
# ---------------------------------------------------------
# Final prediction and save artifacts
# ---------------------------------------------------------
test_probs = test_probs_sum / CFG.N_SPLITS
test_probs_biased = apply_bias(test_probs, final_bias)

oof_pred_raw = np.argmax(oof_probs, axis=1)
oof_pred_biased = np.argmax(oof_probs_biased, axis=1)
test_pred_biased = np.argmax(test_probs_biased, axis=1)

submission = pd.DataFrame({
    CFG.ID_COL: test_raw[CFG.ID_COL],
    CFG.TARGET_COL: pd.Series(test_pred_biased).map(CFG.INV_TARGET_MAP),
})
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

oof_labels = pd.DataFrame({
    CFG.ID_COL: train_raw[CFG.ID_COL],
    "y_true": train_raw[CFG.TARGET_COL],
    "y_pred_raw": pd.Series(oof_pred_raw).map(CFG.INV_TARGET_MAP),
    "y_pred_biased": pd.Series(oof_pred_biased).map(CFG.INV_TARGET_MAP),
})
oof_labels.to_csv(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_labels.csv", index=False)

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_probs)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_probs)
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba_biased.npy", oof_probs_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba_biased.npy", test_probs_biased)
np.save(CFG.SAVE_DIR / "best_class_bias.npy", np.array(final_bias, dtype=np.float32))

feature_columns_df = pd.DataFrame({
    "feature": final_feature_names,
    "is_cat_feature": [c in cat_cols_for_pool for c in final_feature_names],
    "is_pair_te_feature": [str(c).startswith("TE_pair_") for c in final_feature_names],
})
feature_columns_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

avg_importance = np.mean(importances, axis=0)
fi_df = pd.DataFrame({
    "feature": final_feature_names,
    "importance": avg_importance,
}).sort_values("importance", ascending=False)
fi_df.to_csv(CFG.SAVE_DIR / "feature_importance.csv", index=False)

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "raw_cv_ba": raw_cv_ba,
            "refined_biased_cv": final_cv,
            "fold_scores_raw": fold_scores_raw,
            "best_bias": list(map(float, final_bias)),
            "adversarial_auc": float(av_auc),
            "orig_row_weight": CFG.ORIG_ROW_WEIGHT,
            "raw_pair_count": raw_pair_count,
            "accepted_pair_count": len(pair_cols),
            "n_features_before_te_drop": len(features),
            "n_final_features": len(final_feature_names),
            "n_pair_te_features": int(sum(str(c).startswith("TE_pair_") for c in final_feature_names)),
            "threshold_cols_added": THRESHOLD_MIN_COLS,
            "catboost_params": CFG.CATBOOST_PARAMS,
            "bias_search": {
                "stage1": {"range": [-1.0, 1.0], "step": 0.1},
                "stage2": {"center": list(map(float, best_bias_1)), "width": 0.10, "step": 0.02},
                "stage3": {"center": list(map(float, best_bias_2)), "width": 0.02, "step": 0.005},
            },
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

summary_df = pd.DataFrame({
    "item": [
        "adversarial_auc",
        "raw_pair_count",
        "accepted_pair_count",
        "raw_cv_ba",
        "refined_biased_cv",
        "bias_low",
        "bias_medium",
        "bias_high",
        "n_final_features",
    ],
    "value": [
        av_auc,
        raw_pair_count,
        len(pair_cols),
        raw_cv_ba,
        final_cv,
        final_bias[0],
        final_bias[1],
        final_bias[2],
        len(final_feature_names),
    ],
})
summary_df.to_csv(CFG.SAVE_DIR / "summary.csv", index=False)

print("saved to:", CFG.SAVE_DIR)
print(submission[CFG.TARGET_COL].value_counts(normalize=True).sort_index() * 100)
print(summary_df)

saved to: /kaggle/working/exp_20260420_055_cat018_threshold_min_from_family
Irrigation_Need
High       3.770741
Low       59.132222
Medium    37.097037
Name: proportion, dtype: float64
                  item        value
0      adversarial_auc     0.694620
1       raw_pair_count   351.000000
2  accepted_pair_count   315.000000
3            raw_cv_ba     0.977059
4    refined_biased_cv     0.978539
5             bias_low    -1.060000
6          bias_medium    -0.990000
7            bias_high     0.090000
8     n_final_features  1002.000000
